In [1]:
# !pip install netCDF4

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from datetime import datetime

In [2]:
directory = 'Github/NaturalDisasterProject/data/ncei_noaa/'
ext = ('.csv')

file = []

for files in os.listdir(directory):
    if (files.endswith(ext)):
        file.append(files)

file.sort()

pattern = re.compile(r'^StormEvents_locations-ftp_v1\.0_d\d{4}_c\d+\.csv$')
storm_locations = [s for s in file if pattern.match(s)]

pattern = re.compile(r'^StormEvents_details-ftp_v1\.0_d\d{4}_c\d+\.csv$')
storm_details = [s for s in file if pattern.match(s)]

# print(filtered_files)
# print(len(storm_locations))
# print(len(storm_details))
storm_loc_filtered = storm_locations[32:] #only in range 2004-2023

storm_det_filtered = storm_details[54:] #only in range 2004-2023


df = pd.read_csv(directory + storm_det_filtered[0])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52409 entries, 0 to 52408
Data columns (total 51 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BEGIN_YEARMONTH     52409 non-null  int64  
 1   BEGIN_DAY           52409 non-null  int64  
 2   BEGIN_TIME          52409 non-null  int64  
 3   END_YEARMONTH       52409 non-null  int64  
 4   END_DAY             52409 non-null  int64  
 5   END_TIME            52409 non-null  int64  
 6   EPISODE_ID          52409 non-null  int64  
 7   EVENT_ID            52409 non-null  int64  
 8   STATE               52409 non-null  object 
 9   STATE_FIPS          52409 non-null  int64  
 10  YEAR                52409 non-null  int64  
 11  MONTH_NAME          52409 non-null  object 
 12  EVENT_TYPE          52409 non-null  object 
 13  CZ_TYPE             52409 non-null  object 
 14  CZ_FIPS             52409 non-null  int64  
 15  CZ_NAME             52409 non-null  object 
 16  WFO 

In [3]:
df.head()

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,200412,29,1800,200412,30,1200,1182771,5430389,MONTANA,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Heavy snow event across southwest Montana brou...,NaN,PDS
1,200412,29,1800,200412,30,1200,1182771,5430390,MONTANA,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Heavy snow event across southwest Montana brou...,NaN,PDS
2,200412,8,1800,200412,8,1800,1182769,5430387,IDAHO,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A vigorous winter storm brought strong winds a...,NaN,PDS
3,200412,19,1500,200412,19,1700,1182770,5430388,MONTANA,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Prefrontal winds were channeled through east t...,NaN,PDS
4,200412,14,600,200412,14,800,1182772,5430391,MONTANA,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A winter storm with light snow followed by fre...,NaN,PDS


In [4]:
def combine_dates(yearmonth, day):
    yearmonth = yearmonth.astype(str)
    day = day.astype(str)
    y_m_d = yearmonth + day
    return y_m_d

In [5]:
df.insert(0, 'BEGIN_YMD', combine_dates(df['BEGIN_YEARMONTH'], df['BEGIN_DAY']))
df.insert(1, 'END_YMD', combine_dates(df['END_YEARMONTH'], df['END_DAY']))
df = df.drop(columns = ['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME'])
df['BEGIN_YMD'] = pd.to_datetime(df['BEGIN_YMD'], format='%Y%m%d')
df['END_YMD'] = pd.to_datetime(df['END_YMD'], format='%Y%m%d')
df

,BEGIN_YMD,END_YMD,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,2004-12-29,2004-12-30,1182771,5430389,MONTANA,30,2004,December,Heavy Snow,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Heavy snow event across southwest Montana brou...,NaN,PDS
1,2004-12-29,2004-12-30,1182771,5430390,MONTANA,30,2004,December,Heavy Snow,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Heavy snow event across southwest Montana brou...,NaN,PDS
2,2004-12-08,2004-12-08,1182769,5430387,IDAHO,16,2004,December,Winter Storm,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A vigorous winter storm brought strong winds a...,NaN,PDS
3,2004-12-19,2004-12-19,1182770,5430388,MONTANA,30,2004,December,High Wind,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Prefrontal winds were channeled through east t...,NaN,PDS
4,2004-12-14,2004-12-14,1182772,5430391,MONTANA,30,2004,December,Winter Weather,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A winter storm with light snow followed by fre...,NaN,PDS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52404,2004-02-17,2004-02-17,2152282,5432662,NEVADA,32,2004,February,High Wind,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51 knot (59 mph) wind gust reported at Sutcliffe.,PDS
52405,2004-02-17,2004-02-17,2152277,5432657,NEVADA,32,2004,February,High Wind,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59 knot (68 mph) wind gust reported at Hawthorne.,PDS
52406,2004-02-17,2004-02-17,2152278,5432658,NEVADA,32,2004,February,High Wind,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53 knot (61 mph) wind gust recorded at Desert ...,PDS
52407,2004-02-06,2004-02-07,2152272,5431352,CALIFORNIA,6,2004,February,Heavy Snow,Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fast-moving winter storm system brought snow t...,NaN,PDS


In [6]:
list(df.columns)

['BEGIN_YMD',
 'END_YMD',
 'EPISODE_ID',
 'EVENT_ID',
 'STATE',
 'STATE_FIPS',
 'YEAR',
 'MONTH_NAME',
 'EVENT_TYPE',
 'CZ_TYPE',
 'CZ_FIPS',
 'CZ_NAME',
 'WFO',
 'BEGIN_DATE_TIME',
 'CZ_TIMEZONE',
 'END_DATE_TIME',
 'INJURIES_DIRECT',
 'INJURIES_INDIRECT',
 'DEATHS_DIRECT',
 'DEATHS_INDIRECT',
 'DAMAGE_PROPERTY',
 'DAMAGE_CROPS',
 'SOURCE',
 'MAGNITUDE',
 'MAGNITUDE_TYPE',
 'FLOOD_CAUSE',
 'CATEGORY',
 'TOR_F_SCALE',
 'TOR_LENGTH',
 'TOR_WIDTH',
 'TOR_OTHER_WFO',
 'TOR_OTHER_CZ_STATE',
 'TOR_OTHER_CZ_FIPS',
 'TOR_OTHER_CZ_NAME',
 'BEGIN_RANGE',
 'BEGIN_AZIMUTH',
 'BEGIN_LOCATION',
 'END_RANGE',
 'END_AZIMUTH',
 'END_LOCATION',
 'BEGIN_LAT',
 'BEGIN_LON',
 'END_LAT',
 'END_LON',
 'EPISODE_NARRATIVE',
 'EVENT_NARRATIVE',
 'DATA_SOURCE']

In [7]:
df['BEGIN_DATE_TIME'][0]

'29-DEC-04 18:00:00'

In [8]:
df.dropna(subset = ['BEGIN_LAT','BEGIN_LON','END_LAT','END_LON'], inplace=True)
df = df.drop(columns = ['MONTH_NAME', 'BEGIN_DATE_TIME', 'END_DATE_TIME'])
df

,BEGIN_YMD,END_YMD,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,EVENT_TYPE,CZ_TYPE,CZ_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
14,2004-12-07,2004-12-07,1182987,5430439,ALABAMA,1,2004,Thunderstorm Wind,C,63,...,NaN,NaN,EUTAW,32.83333,-87.88333,32.83333,-87.88333,NaN,A few trees were blown down along US 11 near E...,PDS
43,2004-12-07,2004-12-07,1182786,5430509,TENNESSEE,47,2004,Thunderstorm Wind,C,111,...,NaN,NaN,LAFAYETTE,36.51667,-86.03333,36.51667,-86.03333,NaN,Report of 2 trees were blown down on a rural c...,PDS
52,2004-08-20,2004-08-20,1183447,5430944,NEVADA,32,2004,Hail,C,15,...,NaN,NaN,BATTLE MTN,40.65000,-116.91667,40.65000,-116.91667,NaN,NaN,PDS
64,2004-12-10,2004-12-10,1182787,5430629,TENNESSEE,47,2004,Hail,C,111,...,5.0,W,LAFAYETTE,36.51667,-86.11667,36.51667,-86.11667,NaN,Hailing so hard afraid it was damaging tractor.,PDS
98,2004-12-07,2004-12-07,1182786,5430618,TENNESSEE,47,2004,Thunderstorm Wind,C,141,...,NaN,NaN,COOKEVILLE,36.16667,-85.50000,36.16667,-85.50000,NaN,TDOT reported a few trees were blown down. One...,PDS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52236,2004-12-09,2004-12-09,1182727,5429504,MISSISSIPPI,28,2004,Thunderstorm Wind,C,127,...,1.0,S,MAGEE,32.76667,-88.65000,32.76667,-88.65000,NaN,Two trees were blown down on Coats Road.,PDS
52307,2004-12-06,2004-12-06,1182738,5429650,TEXAS,48,2004,Hail,C,75,...,5.0,W,CHILDRESS,34.41667,-100.28333,34.41667,-100.28333,NaN,NaN,PDS
52308,2004-12-06,2004-12-06,1182738,5429651,TEXAS,48,2004,Hail,C,75,...,NaN,NaN,CHILDRESS,34.41667,-100.20000,34.41667,-100.20000,NaN,NaN,PDS
52309,2004-12-07,2004-12-07,1182739,5429652,MISSISSIPPI,28,2004,Tornado,C,17,...,2.0,NNE,HOULKA,34.06667,-89.00000,34.06667,-89.00000,NaN,The tornado touched down just northeast of Hou...,PDS


In [9]:
southeastern_states = [
    "TEXAS",
    "LOUISIANA",
    "MISSISSIPPI",
    "ALABAMA",
    "GEORGIA",
    "FLORIDA",
    "SOUTH CAROLINA",
    "NORTH CAROLINA",
    "ARKANSAS",
    "TENNESSEE"
]
df = df[df['STATE'].isin(southeastern_states)]

In [11]:
col_order = list(df.columns)
col_to_move = col_order[-7:-3].copy()
for x in col_to_move:
    col_order.remove(x)
col_order[2:2] = col_to_move
col_order

['BEGIN_YMD',
 'END_YMD',
 'BEGIN_LAT',
 'BEGIN_LON',
 'END_LAT',
 'END_LON',
 'EPISODE_ID',
 'EVENT_ID',
 'STATE',
 'STATE_FIPS',
 'YEAR',
 'EVENT_TYPE',
 'CZ_TYPE',
 'CZ_FIPS',
 'CZ_NAME',
 'WFO',
 'CZ_TIMEZONE',
 'INJURIES_DIRECT',
 'INJURIES_INDIRECT',
 'DEATHS_DIRECT',
 'DEATHS_INDIRECT',
 'DAMAGE_PROPERTY',
 'DAMAGE_CROPS',
 'SOURCE',
 'MAGNITUDE',
 'MAGNITUDE_TYPE',
 'FLOOD_CAUSE',
 'CATEGORY',
 'TOR_F_SCALE',
 'TOR_LENGTH',
 'TOR_WIDTH',
 'TOR_OTHER_WFO',
 'TOR_OTHER_CZ_STATE',
 'TOR_OTHER_CZ_FIPS',
 'TOR_OTHER_CZ_NAME',
 'BEGIN_RANGE',
 'BEGIN_AZIMUTH',
 'BEGIN_LOCATION',
 'END_RANGE',
 'END_AZIMUTH',
 'END_LOCATION',
 'EPISODE_NARRATIVE',
 'EVENT_NARRATIVE',
 'DATA_SOURCE']

In [18]:
for x in col_to_move:
    df[x] = df[x].round().astype(int)
df = df[col_order]
df.head()

,BEGIN_YMD,END_YMD,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
14,2004-12-07,2004-12-07,33,-88,33,-88,1182987,5430439,ALABAMA,1,...,NaN,NaN,NaN,EUTAW,NaN,NaN,EUTAW,NaN,A few trees were blown down along US 11 near E...,PDS
43,2004-12-07,2004-12-07,37,-86,37,-86,1182786,5430509,TENNESSEE,47,...,NaN,NaN,NaN,LAFAYETTE,NaN,NaN,LAFAYETTE,NaN,Report of 2 trees were blown down on a rural c...,PDS
64,2004-12-10,2004-12-10,37,-86,37,-86,1182787,5430629,TENNESSEE,47,...,NaN,5.0,W,LAFAYETTE,5.0,W,LAFAYETTE,NaN,Hailing so hard afraid it was damaging tractor.,PDS
98,2004-12-07,2004-12-07,36,-86,36,-86,1182786,5430618,TENNESSEE,47,...,NaN,NaN,NaN,COOKEVILLE,NaN,NaN,COOKEVILLE,NaN,TDOT reported a few trees were blown down. One...,PDS
99,2004-12-07,2004-12-07,36,-86,36,-86,1182786,5430619,TENNESSEE,47,...,NaN,NaN,NaN,SMITHVILLE,NaN,NaN,SMITHVILLE,NaN,A tree was down on Evans Mill Rd.,PDS


In [32]:
test = df.loc[2002].iloc[:2]
test

BEGIN_YMD    2004-12-06 00:00:00
END_YMD      2004-12-07 00:00:00
Name: 2002, dtype: object

In [34]:
for x in pd.date_range(start=test.iloc[0], end=test.iloc[1]):
    print(x)

2004-12-06 00:00:00
2004-12-07 00:00:00


In [31]:
df[df['BEGIN_YMD'] != df['END_YMD']]

,BEGIN_YMD,END_YMD,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,EVENT_TYPE,CZ_TYPE,CZ_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
2002,2004-12-06,2004-12-07,1183171,5432104,ALABAMA,1,2004,Flash Flood,C,79,...,NaN,NaN,MOULTON,34.25000,-86.68333,34.25000,-86.68333,NaN,Flash flooding initially occurred in Moulton. ...,PDS
2347,2004-12-06,2004-12-07,1183171,5431459,ALABAMA,1,2004,Flash Flood,C,77,...,NaN,NaN,FLORENCE,34.43333,-88.13333,34.43333,-88.13333,NaN,Flash flooding initially occurred in Florence ...,PDS
2348,2004-12-06,2004-12-07,1183171,5431460,ALABAMA,1,2004,Flash Flood,C,83,...,NaN,NaN,ATHENS,34.51667,-87.73333,34.51667,-87.70000,NaN,Flash flooding initially in Athens evolved int...,PDS
2349,2004-12-06,2004-12-07,1183171,5431461,ALABAMA,1,2004,Flash Flood,C,89,...,NaN,NaN,COUNTYWIDE,34.73333,-87.70000,34.73333,-87.70000,NaN,Flash flooding occurred countywide with numero...,PDS
5021,2004-02-05,2004-02-06,1165941,5383440,MISSISSIPPI,28,2004,Flash Flood,C,85,...,NaN,NaN,COUNTYWIDE,31.48333,-90.91667,31.48333,-90.91667,NaN,Five to six inches of rain fell across all of ...,PDS
5091,2004-02-05,2004-02-06,1165941,5383436,MISSISSIPPI,28,2004,Flash Flood,C,29,...,NaN,NaN,COUNTYWIDE,31.63333,-91.05000,31.63333,-91.05000,NaN,Five to seven inches of rain fell across Copia...,PDS
7979,2004-02-05,2004-02-06,1165941,5383836,MISSISSIPPI,28,2004,Flash Flood,C,99,...,NaN,NaN,COUNTYWIDE,32.01667,-89.53333,32.01667,-89.53333,NaN,Five to seven inches of rain fell across Nesho...,PDS
8011,2004-02-05,2004-02-06,1165941,5383830,MISSISSIPPI,28,2004,Flash Flood,C,129,...,NaN,NaN,COUNTYWIDE,31.86667,-89.88333,31.86667,-89.88333,NaN,Five to seven inches of rain fell across Smith...,PDS
9207,2004-02-05,2004-02-06,1165941,5383839,MISSISSIPPI,28,2004,Flash Flood,C,101,...,NaN,NaN,COUNTYWIDE,32.38333,-89.11667,32.38333,-89.11667,NaN,Five to seven inches of rain fell across Newto...,PDS
9238,2004-02-05,2004-02-06,1165941,5383823,MISSISSIPPI,28,2004,Flash Flood,C,69,...,NaN,NaN,COUNTYWIDE,31.86667,-90.18333,31.86667,-90.18333,NaN,Five to seven inches of rain fell across Kempe...,PDS
